# Evaluate external diagnosis dates

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from smmnlp.evaluation import calculate_binary_metrics, print_metrics
from smmnlp.paths import ProjectPaths

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Load reference and reviewed model dates


In [ ]:
reference_dates = pd.read_excel(
    paths.data("Update/ExternalValidation/True/external_sample100_notes_true_S2.xlsx")
)[["pat_mrn_id", "smm_date_true"]]
model_dates = pd.read_excel(
    paths.data("Update/ExternalValidation/Results/EndToEnd_dxdate_human_final.xlsx")
)[["pat_mrn_id", "DxDate_LLM_Human"]]
date_results = reference_dates.merge(model_dates, on="pat_mrn_id", how="left")
date_results["reference_date"] = pd.to_datetime(
    date_results["smm_date_true"].replace("unknown", pd.NA), format="%m/%d/%Y"
)
date_results["predicted_date"] = pd.to_datetime(
    date_results["DxDate_LLM_Human"].replace("unknown", pd.NA), format="%m/%d/%Y"
)


## Evaluate date presence across the complete reference cohort

Missing model dates remain in the denominator.


In [ ]:
metric_result = calculate_binary_metrics(
    date_results["reference_date"].notna().astype(int),
    date_results["predicted_date"].notna().astype(int),
)
print_metrics(metric_result)
ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm,
    display_labels=["Date absent", "Date present"],
).plot()
plt.show()


## Compare available date pairs


In [ ]:
paired_dates = date_results.loc[
    date_results["reference_date"].notna() & date_results["predicted_date"].notna()
].copy()
paired_dates["date_difference_days"] = (
    paired_dates["predicted_date"] - paired_dates["reference_date"]
).dt.days
plt.figure(figsize=(8, 5))
plt.hist(paired_dates["date_difference_days"], bins=30, color="orange", edgecolor="black")
plt.xlabel("Predicted date minus reference date (days)")
plt.ylabel("Frequency")
plt.title("External diagnosis date differences")
plt.show()
output_path = Path(paths.data("Update/ExternalValidation/EndToEnd_ExternalValidation_datediff_human.xlsx"))
output_path.parent.mkdir(parents=True, exist_ok=True)
paired_dates["date_difference_days"].to_excel(
    output_path, engine="xlsxwriter", sheet_name="Sheet1", index=False
)
